In [2]:
import os
import json
import tensorflow as tf
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# --- STEP 1: DATASET AUDIT (The "Anti-Crash" Logic) ---
# This ensures no corrupted files stop the training process
data_dir = '../data/web_scraped'

def clean_dataset(directory):
    removed_count = 0
    print("🔍 Auditing diamond images for corruption...")
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                file_path = os.path.join(root, file)
                try:
                    with Image.open(file_path) as img:
                        img.verify() 
                except Exception:
                    print(f"❌ Removing corrupted/invalid file: {file_path}")
                    os.remove(file_path)
                    removed_count += 1
    return removed_count

bad_files = clean_dataset(data_dir)
print(f"✨ Audit complete. Removed {bad_files} files. Ready for training.")

# --- STEP 2: IMAGE PREPROCESSING & AUGMENTATION ---
# Optimized for real-world photos and complex geometries (Marquise/Cushion)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,      # High rotation handles non-standard photo angles
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.25,       # Increased to better detect Cushion corners & Marquise points
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=128,
    class_mode='categorical',
    subset='training'
)

# --- STEP 3: MODEL ARCHITECTURE (MobileNetV2) ---
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x) 
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Transfer Learning: Freeze base layers to speed up training on standard laptops
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# --- STEP 4: TRAINING ---
print("🚀 Starting training...")
model.fit(train_generator, epochs=10)

# --- STEP 5: SAVING ASSETS ---
# We save these in '../models' so app.py can find them from the root folder
if not os.path.exists('../models'):
    os.makedirs('../models')

model.save('../models/shape_model.h5')

# Crucial: Save class indices so the app knows which index = which diamond shape
with open('../models/classes.json', 'w') as f:
    json.dump(train_generator.class_indices, f)

print("✅ Training complete! Models and class labels saved to /models folder.")

🔍 Auditing diamond images for corruption...
✨ Audit complete. Removed 0 files. Ready for training.
Found 39014 images belonging to 8 classes.
🚀 Starting training...
Epoch 1/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1424s 5s/step - accuracy: 0.9710 - loss: 0.0964
Epoch 2/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1368s 4s/step - accuracy: 0.9895 - loss: 0.0375
Epoch 3/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1390s 5s/step - accuracy: 0.9896 - loss: 0.0353
Epoch 4/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 12582s 41s/step - accuracy: 0.9926 - loss: 0.0290
Epoch 5/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1989s 7s/step - accuracy: 0.9924 - loss: 0.0267
Epoch 6/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 2333s 8s/step - accuracy: 0.9931 - loss: 0.0255
Epoch 7/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1387s 5s/step - accuracy: 0.9939 - loss: 0.0229
Epoch 8/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1659s 5s/step - accuracy: 0.9946 - loss: 0.0214
Epoch 9/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 1473s 5s/step - accuracy: 0.9940 - loss: 0.0211
Epoch 10/10
305/305 ━━━━━━━━━━━━━━━━━━━━ 

✅ Training complete! Models and class labels saved to /models folder.
